# 01 — Validation Zilinskas + EDA temporelle

**Objectif** : Sanity-check complet du parquet `style-control-analysis/battles_bt_styled.parquet` :
1. Reproduire les coefficients BT style-controlled de Zilinskas (bold ≈ +19%, lists ≈ +18%, headers ≈ +16%)
2. Explorer la structure du dataset (modèles, distribution temporelle si disponible)
3. Identifier ce qu'il manque pour R1/R2bis/R4 (→ besoin de timestamps depuis HF)

**Ne requiert pas de `.env` ni de download HF.**

**Output** : confirmation visuelle du sanity check + liste des actions suivantes.

In [ ]:
# Cell 1 — Imports
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ROOT    = Path('..')
ZILINSKAS = ROOT / 'style-control-analysis'
DATA    = ROOT / 'data'
FIGURES = ROOT / 'paper' / 'figures'

RANDOM_STATE = 42
STYLE_FEATS  = ['headers', 'lists', 'bold', 'code_blocks', 'emoji']

print('Imports OK')

In [ ]:
# Cell 2 — Chargement + aperçu schéma
df = pd.read_parquet(ZILINSKAS / 'battles_bt_styled.parquet')

print(f'Shape          : {df.shape}')
print(f'Colonnes       : {df.columns.tolist()}')
print(f'winner         : {df.winner.value_counts().to_dict()}')
print(f'source         : {df.source.value_counts().to_dict()}')
print(f'Modèles uniques: {len(set(df.model_a_name) | set(df.model_b_name))}')
print()

# ⚠️ Pas de colonne date/timestamp dans ce parquet — conséquence notée ci-dessous
has_date = any('time' in c.lower() or 'date' in c.lower() for c in df.columns)
print(f'Timestamp présent dans le parquet : {has_date}')
print('=> Pour R1/R2bis/R4, il faudra joindre les dates depuis HF votes (comparia-votes).')

In [ ]:
# Cell 3 — Distribution des features de style
fig, axes = plt.subplots(1, len(STYLE_FEATS), figsize=(16, 4))
for ax, feat in zip(axes, STYLE_FEATS):
    all_vals = pd.concat([df[f'{feat}_a'], df[f'{feat}_b']])
    # Tronquer à P99 pour la lisibilité
    cap = all_vals.quantile(0.99)
    ax.hist(all_vals.clip(upper=cap), bins=30, color='steelblue', alpha=0.7)
    ax.set_title(f'{feat}\n(P99={cap:.0f})')
    ax.set_xlabel('Compte brut')
    pct_zero = (all_vals == 0).mean() * 100
    ax.annotate(f'{pct_zero:.0f}% zéros', xy=(0.6, 0.85), xycoords='axes fraction', fontsize=9)

plt.suptitle('Distribution des features style dans battles_bt_styled.parquet', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'EDA_style_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée.')

In [ ]:
# Cell 4 — Sanity check BT style-controlled (méthode exacte Zilinskas)
#
# Réf : compute_bt_style_controlled() dans style-control-analysis/clean_and_analyze.py
# y = 0/1 (pas de miroir), X_model = +1/-1, X_style = delta_brut standardisé

dec = df[df['winner'].isin(['model_a', 'model_b'])].copy().reset_index(drop=True)

models = sorted(set(dec.model_a_name) | set(dec.model_b_name))
m2i    = {m: i for i, m in enumerate(models)}
nm, N  = len(models), len(dec)

ai = dec.model_a_name.map(m2i).values
bi = dec.model_b_name.map(m2i).values
y  = (dec.winner == 'model_a').astype(float).values

X_model = np.zeros((N, nm))
X_model[np.arange(N), ai] = 1
X_model[np.arange(N), bi] = -1

X_style = (
    dec[[f'{f}_a' for f in STYLE_FEATS]].values
    - dec[[f'{f}_b' for f in STYLE_FEATS]].values
).astype(float)
X_style = StandardScaler().fit_transform(X_style)

X  = np.hstack([X_model, X_style])
lr = LogisticRegression(fit_intercept=False, penalty=None, max_iter=5000)
lr.fit(X, y)

style_coefs = dict(zip(STYLE_FEATS, lr.coef_[0][nm:]))

REF = {'headers': 15.6, 'lists': 18.0, 'bold': 19.0, 'code_blocks': 0.0, 'emoji': 0.0}
print(f'Battles analysés : {N:,}  |  modèles : {nm}')
print(f'{"Feature":12s}  {"Nos coefs (%/SD)":>18s}  {"Ref Zilinskas (%/SD)":>22s}  {"Delta":>8s}')
print('-' * 65)
for f, c in style_coefs.items():
    pct = (np.exp(c) - 1) * 100
    ref = REF.get(f, 0.0)
    print(f'{f:12s}  {pct:+17.1f}%  {ref:+21.1f}%  {pct-ref:+7.1f}%')

In [ ]:
# Cell 5 — Forest plot des coefficients avec comparaison Zilinskas
feats = list(style_coefs.keys())
pcts  = [(np.exp(c) - 1) * 100 for c in style_coefs.values()]
refs  = [REF.get(f, 0.0) for f in feats]

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(feats))
ax.barh(y_pos - 0.2, pcts,  height=0.35, color='steelblue', label='Nos coefs (toutes sources)')
ax.barh(y_pos + 0.2, refs,  height=0.35, color='darkorange', alpha=0.7, label='Zilinskas (votes seuls)')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(feats, fontsize=11)
ax.set_xlabel('Effet sur la probabilité de victoire (% / écart-type)', fontsize=10)
ax.set_title('Sanity check — BT style-controlled\nComparaison avec Zilinskas (2026)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'EDA_sanity_check_BT.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sanity check OK — pipeline BT validé.')

In [ ]:
# Cell 6 — Top 20 modèles par volume de battles (pour évaluer la couverture R2bis)
model_counts = (
    pd.concat([dec.model_a_name, dec.model_b_name])
    .value_counts()
    .head(20)
)
print('Top 20 modèles par nb de battles décisifs:')
print(model_counts.to_string())

# Modèles avec ≥ 500 battles (seuil min pour une cohorte BT mensuelle stable)
seuil = 500
n_suf = (pd.concat([dec.model_a_name, dec.model_b_name]).value_counts() >= seuil).sum()
print(f'\nModèles avec ≥ {seuil} battles : {n_suf}')

In [ ]:
# Cell 7 — Résumé des besoins pour la suite
print('=' * 60)
print('RÉSUMÉ : ce qu\'il manque pour chaque module')
print('=' * 60)
print()
print('R1 — Convergence stylistique temporelle')
print('  ✅ Parquet Zilinskas (features style 5D)')
print('  ❌ Timestamps → joindre depuis HF comparia-votes (~50 Mo)')
print('  ❌ Features additionnelles (spaCy, n_tokens…) → HF conversations (~9 Go)')
print()
print('R2bis — Style Premium longitudinal')
print('  ✅ Pipeline BT validé')
print('  ❌ Timestamps → même besoin que R1')
print()
print('R3 — Preuve causale par contrefactuels')
print('  ❌ Textes des réponses → HF conversations')
print('  ❌ API Mistral (MISTRAL_API_KEY dans .env)')
print()
print('R4 — Forecast d\'effondrement')
print('  ❌ Série temporelle D_t → dépend de R1')
print()
print('PROCHAINE ÉTAPE : remplir .env + lancer check_access.py')
print('  => python scripts/check_access.py')
print('  => Si HF OK : joindre timestamps depuis comparia-votes (streaming, ~50 Mo)')